# DEPURADO DE DATOS

Lo primero es leer los xlsx, el primero tiene una una unica hoja, llamaremos a esta tabla transfermarkt. En el segundo utilizaremos las 5 hojas del xlsx, las llamaremos whoscore_ofernsivo, whoscore_defensivo, whoscore_distribucion, whoscore_xG y equipos.

In [22]:
import pandas as pd
from IPython.display import display, Markdown
import numpy as np
import re
import unicodedata
from pathlib import Path


In [23]:
# === CONFIGURACIÓN ===
PROJECT_ROOT = Path.cwd()
archivo_transfermarkt = PROJECT_ROOT / "DATOS TRANSFERMARKET.xlsx"
archivo_whoscored = PROJECT_ROOT / "whoscored_todas_ligas_jugadores_2025_26.xlsx"
archivo_posiciones_valormax = PROJECT_ROOT / "transfermarkt_7_ligas_profiles_incremental_2025_26.csv"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

CONFIG = {
    "archivo_transfermarkt":        archivo_transfermarkt,
    "archivo_whoscored":            archivo_whoscored,
    "archivo_posiciones_valormax":  archivo_posiciones_valormax,
    "nombre_hoja_final":            "equipos",
    "prefijo_whoscored":            "whoscore_",
}

In [24]:
def cargar_transfermarkt(ruta: str) -> pd.DataFrame:
    df = pd.read_excel(ruta)
    print(f"transfermarkt: {df.shape[0]} filas, {df.shape[1]} columnas")
    return df


def cargar_whoscored(ruta: str, prefijo: str, nombre_final: str) -> dict:
    hojas = pd.read_excel(ruta, sheet_name=None)
    nombres = list(hojas.keys())
    datos = {}

    for i, nombre_hoja in enumerate(nombres):
        nuevo = nombre_final if i == len(nombres) - 1 else f"{prefijo}{nombre_hoja}"
        datos[nuevo] = hojas[nombre_hoja]
        print(f"{nuevo}: {hojas[nombre_hoja].shape[0]} filas, {hojas[nombre_hoja].shape[1]} columnas")

    return datos

def cargar_posiciones_valormax(ruta: str) -> pd.DataFrame:
    df = pd.read_csv(ruta)
    print(f"posiciones_valormax: {df.shape[0]} filas, {df.shape[1]} columnas")
    return df

Primero debemos eliminar los duplicados, ya que se estan duplicando algunas filas debido a que el nombre del club sale de manera diferente (Ej. REAL MADRID/REAL MADRID CF)

In [25]:
def eliminar_duplicados_ignorando_col_a(df: pd.DataFrame) -> pd.DataFrame:
    antes = len(df)
    cols = df.columns[1:].tolist()
    df = df.drop_duplicates(subset=cols, keep="first").reset_index(drop=True)
    print(f"Duplicados eliminados: {antes} -> {len(df)} ({antes - len(df)} filas)")
    return df

En segundo lugar creamos una funcion que elimine columnas:

In [26]:
def eliminar_columnas(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    existentes = [c for c in columnas if c in df.columns]
    faltantes  = [c for c in columnas if c not in df.columns]
    df = df.drop(columns=existentes)
    print(f"Columnas eliminadas: {existentes}")
    if faltantes:
        print(f"No encontradas (se omiten): {faltantes}")
    return df

Aqui se implementa una logica para extraer el anterior club de la columna que viene el el formato: Chelsea FC: Ablöse 35,00 mill. € | Chelsea FC

In [27]:
def limpiar_previous_club(df: pd.DataFrame, columna: str = "tm_previous_club") -> pd.DataFrame:
    """Deja en la columna solo el nombre del club anterior (último segmento tras '|')."""
    def extraer(valor):
        if pd.isna(valor) or str(valor).strip() == "":
            return np.nan
        texto = str(valor).strip()
        return texto.split("|")[-1].strip() if "|" in texto else np.nan

    df[columna] = df[columna].apply(extraer)
    print(f"Columna '{columna}' limpiada")
    return df

- Elimina la columna 'Jugador'.
      - Separa 'Jugador.1' (formato 'Nombre Edad, Posiciones') en
        tres columnas: nombre, edad, posiciones.
      - Lo que no cumpla el patrón se deja vacío.
      - Coloca las columnas nuevas justo detrás de 'ws_tab'.
      - Elimina la columna original 'Jugador.1'.

In [28]:
def limpiar_hoja_whoscored(df: pd.DataFrame, nombre_hoja: str) -> pd.DataFrame:
    """
    Para hojas de WhoScored:
      - Elimina la columna 'Jugador'.
      - Separa 'Jugador.1' (formato 'Nombre Edad, Posiciones') en
        tres columnas: nombre, edad, posiciones.
      - Lo que no cumpla el patrón se deja vacío.
      - Coloca las columnas nuevas justo detrás de 'ws_tab'.
      - Elimina la columna original 'Jugador.1'.
    """
    if "Jugador" in df.columns:
        df = df.drop(columns=["Jugador"])

    patron = re.compile(r"^(.+?)\s+(\d{1,2}),\s*(.+)$")

    def extraer(valor):
        if pd.isna(valor):
            return pd.Series([np.nan, np.nan, np.nan])
        match = patron.match(str(valor).strip())
        if match:
            return pd.Series([match.group(1).strip(), int(match.group(2)), match.group(3).strip()])
        return pd.Series([np.nan, np.nan, np.nan])

    if "Jugador.1" in df.columns:
        df[["nombre", "edad", "posiciones"]] = df["Jugador.1"].apply(extraer)
        df["edad"] = df["edad"].astype("Int64")
        df = df.drop(columns=["Jugador.1"])

        # Reordenar: nombre, edad, posiciones justo detrás de ws_tab
        if "ws_tab" in df.columns:
            cols = [c for c in df.columns if c not in ["nombre", "edad", "posiciones"]]
            idx = cols.index("ws_tab") + 1
            nuevo_orden = cols[:idx] + ["nombre", "edad", "posiciones"] + cols[idx:]
            df = df[nuevo_orden]

        # Reporte de vacíos
        vacios = df[["nombre", "edad", "posiciones"]].isna().sum()
        print(f"{nombre_hoja}:  nombre={vacios['nombre']}  edad={vacios['edad']}  posiciones={vacios['posiciones']}")

    return df

Aqui normalizamos la columna de partidos jugados, separandolos en titulares y suplente.

In [29]:
def separar_jgdos(df: pd.DataFrame, columna: str = "Jgdos") -> pd.DataFrame:
    """
    Separa la columna 'Jgdos' (formato 'N' o 'N(M)') en:
      - titular:  el número fuera del paréntesis.
      - suplente: el número dentro del paréntesis (0 si no hay paréntesis).
    Lo que no cumpla el patrón se deja vacío. Elimina la columna original.
    Las nuevas columnas ocupan la posición que tenía 'Jgdos'.
    """
    if columna not in df.columns:
        return df

    patron = re.compile(r"^(\d+)(?:\((\d+)\))?$")

    def extraer(valor):
        if pd.isna(valor):
            return pd.Series([pd.NA, pd.NA])
        match = patron.match(str(valor).strip())
        if match:
            titular = int(match.group(1))
            suplente = int(match.group(2)) if match.group(2) else 0
            return pd.Series([titular, suplente])
        return pd.Series([pd.NA, pd.NA])

    df[["titular", "suplente"]] = df[columna].apply(extraer)
    df["titular"]  = df["titular"].astype("Int64")
    df["suplente"] = df["suplente"].astype("Int64")

    # Reemplazar Jgdos por titular y suplente conservando la posición
    nuevo_orden = []
    for c in df.columns:
        if c == columna:
            nuevo_orden.extend(["titular", "suplente"])
        elif c not in ["titular", "suplente"]:
            nuevo_orden.append(c)
    df = df[nuevo_orden]

    return df

Esta funcion elimina todas las columnas que empiezan por Goles, ya que se han generado 90 columnas vacias: 

In [30]:
def eliminar_columnas_por_prefijo(df: pd.DataFrame, prefijo: str) -> pd.DataFrame:
    """Elimina todas las columnas cuyo nombre empieza por el prefijo dado."""
    eliminar = [c for c in df.columns if c.startswith(prefijo)]
    df = df.drop(columns=eliminar)
    print(f"  Columnas eliminadas con prefijo '{prefijo}': {len(eliminar)}")
    return df

Elimino las filas que tengan el nombre del jugador vacio:

In [31]:
def eliminar_filas_vacias_en_columna(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """Elimina las filas en las que la columna indicada está vacía."""
    if columna not in df.columns:
        print(f"  Columna '{columna}' no encontrada, se omite")
        return df
    antes = len(df)
    df = df.dropna(subset=[columna]).reset_index(drop=True)
    print(f"  Filas eliminadas por '{columna}' vacía: {antes - len(df)}  ({antes} -> {len(df)})")
    return df

Aqui normalizamos la columna posicion secundaria, hacemos una logica que vaya añadiendo poisciones, que estan separadas por |, las añade en columnas y la logica para cuando se encuentra con: 

Nombre en país de origen:, Nombre completo,  F. Nacim./Edad:, Lugar de nac.

In [32]:
def separar_secondary_positions(df: pd.DataFrame, columna: str = "tm_secondary_positions") -> pd.DataFrame:
    """
    Extrae las posiciones secundarias de 'tm_secondary_positions'.
    Formato: 'Posición1 | Posición2 | ... | Marcador: | ...'
    Recoge segmentos hasta encontrar un marcador (segmento que termina en ':'
    o que es 'Nombre completo'). Crea pos_sec_1, pos_sec_2, ... y elimina la original.
    """
    if columna not in df.columns:
        return df

    def parsear(valor):
        if pd.isna(valor) or str(valor).strip() == "":
            return []
        posiciones = []
        for seg in str(valor).split("|"):
            seg = seg.strip()
            if not seg:
                continue
            seg_low = seg.lower()
            if seg_low.endswith(":") or seg_low == "nombre completo":
                break
            posiciones.append(seg)
        return posiciones

    listas = df[columna].apply(parsear)
    max_pos = int(listas.map(len).max()) if len(listas) > 0 else 0

    nuevas_columnas = []
    for i in range(max_pos):
        nombre_col = f"pos_sec_{i+1}"
        df[nombre_col] = listas.map(lambda lst, i=i: lst[i] if i < len(lst) else pd.NA)
        nuevas_columnas.append(nombre_col)

    # Reordenar reemplazando la columna original por las nuevas (la original queda fuera)
    nuevo_orden = []
    for c in df.columns:
        if c == columna:
            nuevo_orden.extend(nuevas_columnas)
        elif c not in nuevas_columnas:
            nuevo_orden.append(c)
    df = df[nuevo_orden]

    print(f"  Columnas creadas: {nuevas_columnas}  (máximo {max_pos} posiciones por jugador)")
    return df

Con la ayuda de la inteligencia artificial y mediante tablas de apoyo, se ha llegado a los siguientes matcheos de nombres entre las tablas de Whoscored y las tablas de transfermarkt, los demas que no se han encontrado se eliminarán, ya que se tratan de jugadores que han empezado la temporada en un equipo de las ligas seleccionadas, y la han acabado en un equipo que no se tiene en cuenta en la base de datos.

In [33]:
RENOMBRES_JUGADORES = {
    # clave: nombre tal como aparece en WhoScored (normalizado: minúsculas, sin tildes)
    # valor: nombre tal como aparece en transfermarkt
    "alex amorim":          "amorim",
    "sam szmodics":         "Sammie Szmodics",
    "sidny cabral":         "Sidny Lopes Cabral",
    "tiago silva":          "thiago silva",
    "valentin castellanos": "Taty Castellanos",
    "yang min-hyeok":       "Min-hyeok Yang",
    "vitalii mykolenko":      "Vitaliy Mykolenko",
    "joakim mæhle":           "Joakim Maehle",
    "ferdi kadioglu":         "Ferdi Kadıoğlu",
    "kenan yildiz":           "Kenan Yıldız",
    "altay bayindir":         "Altay Bayındır",
    "frederik rønnow":        "Frederik Rönnow",
    "andrii lunin":           "Andriy Lunin",
    "semih kiliçsoy":         "Semih Kılıçsoy",
    "demir ege tiknaz":       "Demir Ege Tıknaz",
    "maksym talovierov":      "Maksym Taloverov",
    "heorhii sudakov":        "Georgiy Sudakov",
    "illia zabarnyi":         "Ilya Zabarnyi",
    "ameen al dakhil":        "Ameen Al-Dakhil",
    "jones el abdellaoui":    "Jones El-Abdellaoui",
    "al musrati":             "Moatasem Al-Musrati",

    # Orden nombre/apellido invertido (jugadores coreanos)
    "bae jun-ho":             "Jun-ho Bae",
    "hwang hee-chan":         "Hee-chan Hwang",
    "paik seung-ho":          "Seung-ho Paik",
    "eom ji-sung":            "Ji-sung Eom",
    "jeon jin-woo":           "Jin-woo Jeon",
    "djené dakonam":          "Dakonam Djené",

    # Apodos o forma corta/larga del mismo nombre
    "trincão":                "Francisco Trincão",
    "toti gomes":             "Toti",
    "beraldo":                "Lucas Beraldo",
    "ben brereton":           "Ben Brereton Díaz",
    "barbero":                "Iván Barbero",
    "copete":                 "José Copete",
    "carlos ponck":           "Ponck",
    "hannibal mejbri":        "Hannibal",
    "francisco chissumba":    "Chissumba",
    "franco umeh-chibueze":   "Franco Umeh",
    "kialonda gaspar":        "Gaspar",
    "djiamgone jocelin ta bi": "Jocelin Ta Bi",
    "silko-amari thomas":     "Silko Thomas",
    "étienne youté":          "Étienne Youté Kinkoué",
    "thórir helgason":        "Thórir Jóhann Helgason",
    "ollie scarles":          "Oliver Scarles",
    "lionel mpasi":           "Lionel Mpasi-Nzau",
    "sindre egeli":           "Sindre Walle Egeli",
    "syd agina":              "Sydney Agina",
    "max wöber":              "Maximilian Wöber",
    "tasos douvikas":         "Anastasios Douvikas",
    "sam amo-ameyaw":         "Samuel Amo-Ameyaw",
    "omar haktab traoré":     "Omar Traoré",
    "cezary pawet miszta":    "Cezary Miszta",
    "mousa al tamari":        "Mousa Tamari",
    "hevertton santos":       "Hevertton",
    "kostas tsimikas":        "Konstantinos Tsimikas",
    "mikael ellertsson":      "Mikael Egill Ellertsson",
    "adrian barisic":         "Adrian Leon Barisic",
    "cristian cásseres":      "Cristian Cásseres Jr.",
    "raees bangura-williams": "Ra'ees Bangura-Williams",
    "hákon haraldsson":       "Hákon Arnar Haraldsson",
    "abner vinícius":         "Abner",
    "andy robertson":         "Andrew Robertson",
    "pedro trigueira":        "Trigueira",
    "derry john murkin":      "Derry Murkin",

    # Versiones cortas de nombres ingleses comunes
    "benjamín cremaschi":     "Benja Cremaschi",
    "benjamin whiteman":      "Ben Whiteman",
    "charly alcaraz":         "Carlos Alcaraz",
    "jon olasagasti":         "Jon Ander Olasagasti",
    "dan bentley":            "Daniel Bentley",
    "dan mcnamara":           "Danny McNamara",
    "robert brady":           "Robbie Brady",
    "oliver bostock":         "Ollie Bostock",
    "oliver mcburnie":        "Oli McBurnie",
    "dani carvajal":          "Daniel Carvajal",
    "daniel figueira":        "Dani Figueira",
    "jonny":                  "Jonny Otto",
    "thomas cannon":          "Tom Cannon",
    "pacha espino":           "Alfonso Espino",
}


In [34]:
def normalizar(texto):
    """Pasa a minúsculas, quita tildes y espacios sobrantes."""
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def renombrar_jugadores(df: pd.DataFrame, mapeo: dict, columna: str = "nombre") -> pd.DataFrame:
    """
    Renombra jugadores en la columna indicada usando un diccionario.
    Las claves del mapeo se comparan en formato normalizado (sin tildes, minúsculas).
    """
    if columna not in df.columns or not mapeo:
        return df

    mapeo_norm = {normalizar(k): v for k, v in mapeo.items()}

    def reemplazar(valor):
        if pd.isna(valor):
            return valor
        return mapeo_norm.get(normalizar(valor), valor)

    antes = df[columna].copy()
    df[columna] = df[columna].apply(reemplazar)
    cambios = (antes != df[columna]).sum()
    print(f"  Jugadores renombrados: {cambios}")
    return df

Ahora eliminamos los jugadores que no esten en ambas tablas, es decir que sus nombres no coincidan exactamente:

In [35]:
def filtrar_por_interseccion(df: pd.DataFrame, nombres_validos: set, columna: str) -> pd.DataFrame:
    """Deja solo las filas cuyo valor normalizado en 'columna' esté en 'nombres_validos'."""
    if columna not in df.columns:
        return df
    antes = len(df)
    mask = df[columna].map(normalizar).isin(nombres_validos)
    df = df[mask].reset_index(drop=True)
    print(f"  Filas conservadas: {len(df)}/{antes}  (eliminadas: {antes - len(df)})")
    return df

Hemos sacado los jugadores duplicados en las tablas de whoscored (jugadores que han comenzado la temporada en un equipo y la han acabado en otro), como necesito que estos jugadores aparezcan en su actual equipo, hago match por nombres con los jugadores de transfermarkt y su equipo actual. Este proceso nos propone un nuevo problema, los equipos de ambas tablas no matchean perfectamente, entonces se sigue el mismo proceso que con los jugadores, mediante tablas auxiliares y la IA se procede a matchear los siguientes equipos:

In [36]:
RENOMBRES_EQUIPOS = {
    "AC Milan":              "AC Milan",
    "AVS Futebol SAD":       "Avs Futebol",
    "Alverca":               "FC Alverca",
    "Angers":                "Angers SCO",
    "Arouca":                "FC Arouca",
    "Arsenal":               "Arsenal FC",
    "Aston Villa":           "Aston Villa",
    "Atalanta":              "Atalanta de Bérgamo",
    "Athletic Club":         "Athletic Club",
    "Atletico Madrid":       "Atlético de Madrid",
    "Augsburg":              "FC Augsburgo",
    "Auxerre":               "AJ Auxerre",
    "Barcelona":             "FC Barcelona",
    "Bayer Leverkusen":      "Bayer 04 Leverkusen",
    "Bayern Munich":         "Bayern Múnich",
    "Benfica":               "SL Benfica",
    "Birmingham":            "Birmingham City",
    "Blackburn":             "Blackburn Rovers",
    "Bologna":               "Bolonia",
    "Borussia Dortmund":     "Borussia Dortmund",
    "Borussia M.Gladbach":   "Borussia Mönchengladbach",
    "Bournemouth":           "AFC Bournemouth",
    "Braga":                 "SC Braga",
    "Brentford":             "Brentford FC",
    "Brest":                 "Stade Brestois 29",
    "Brighton":              "Brighton & Hove Albion",
    "Bristol City":          "Bristol City",
    "Burnley":               "Burnley FC",
    "Cagliari":              "Cagliari",
    "Casa Pia AC":           "Casa Pia AC",
    "Celta Vigo":            "RC Celta de Vigo",
    "Charlton":              "Charlton Athletic",
    "Chelsea":               "Chelsea FC",
    "Como":                  "Como 1907",
    "Coventry":              "Coventry City",
    "Cremonese":             "US Cremonese",
    "Crystal Palace":        "Crystal Palace",
    "Deportivo Alaves":      "Deportivo Alavés",
    "Derby":                 "Derby County",
    "Eintracht Frankfurt":   "Eintracht Fráncfort\u200b\u200b",
    "Elche":                 "Elche CF",
    "Espanyol":              "RCD Espanyol",
    "Estoril":               "GD Estoril Praia",
    "Estrela da Amadora":    "CF Estrela Amadora",
    "Everton":               "Everton FC",
    "FC Heidenheim":         "1.FC Heidenheim 1846",
    "FC Koln":               "FC Colonia",
    "FC Porto":              "FC Oporto",
    "Famalicao":             "FC Famalicão",
    "Fiorentina":            "Fiorentina",
    "Freiburg":              "SC Friburgo",
    "Fulham":                "Fulham FC",
    "Genoa":                 "Génova",
    "Getafe":                "Getafe CF",
    "Gil Vicente":           "Gil Vicente FC",
    "Girona":                "Girona FC",
    "Hamburger SV":          "Hamburgo SV",
    "Hoffenheim":            "TSG 1899 Hoffenheim",
    "Hull":                  "Hull City",
    "Inter":                 "Inter de Milán",
    "Ipswich":               "Ipswich Town",
    "Juventus":              "Juventus de Turín",
    "Lazio":                 "SS Lazio",
    "Le Havre":              "Le Havre AC",
    "Lecce":                 "US Lecce",
    "Leeds":                 "Leeds United",
    "Leicester":             "Leicester City",
    "Lens":                  "RC Lens",
    "Levante":               "Levante UD",
    "Lille":                 "LOSC Lille",
    "Liverpool":             "Liverpool FC",
    "Lorient":               "FC Lorient",
    "Lyon":                  "Olympique de Lyon",
    "Mainz 05":              "1.FSV Mainz 05",
    "Mallorca":              "RCD Mallorca",
    "Manchester City":       "Manchester City",
    "Manchester United":     "Manchester United",
    "Marseille":             "Olympique de Marsella",
    "Metz":                  "FC Metz",
    "Middlesbrough":         "Middlesbrough FC",
    "Millwall":              "Millwall FC",
    "Monaco":                "AS Mónaco",
    "Moreirense":            "Moreirense FC",
    "Nacional":              "CD Nacional",
    "Nantes":                "FC Nantes",
    "Napoli":                "SSC Nápoles",
    "Newcastle":             "Newcastle United",
    "Nice":                  "OGC Niza",
    "Norwich":               "Norwich City",
    "Nottingham Forest":     "Nottingham Forest",
    "Osasuna":               "CA Osasuna",
    "Oxford":                "Oxford United",
    "Paris FC":              "Paris FC",
    "Paris Saint-Germain":   "París Saint-Germain FC",
    "Parma Calcio 1913":     "Parma",
    "Pisa":                  "Pisa Sporting Club",
    "Portsmouth":            "Portsmouth FC",
    "Preston":               "Preston North End",
    "Queens Park Rangers":   "Queens Park Rangers",
    "RB Leipzig":            "RB Leipzig",
    "Rayo Vallecano":        "Rayo Vallecano",
    "Real Betis":            "Real Betis Balompié",
    "Real Madrid":           "Real Madrid CF",
    "Real Oviedo":           "Real Oviedo",
    "Real Sociedad":         "Real Sociedad",
    "Rennes":                "Stade Rennais FC",
    "Rio Ave":               "Rio Ave FC",
    "Roma":                  "AS Roma",
    "Santa Clara":           "CD Santa Clara",
    "Sassuolo":              "US Sassuolo",
    "Sevilla":               "Sevilla FC",
    "Sheffield United":      "Sheffield United",
    "Sheffield Wednesday":   "Sheffield Wednesday",
    "Southampton":           "Southampton FC",
    "Sporting CP":           "Sporting de Lisboa",
    "St. Pauli":             "FC St. Pauli",
    "Stoke":                 "Stoke City",
    "Strasbourg":            "Racing Club de Estrasburgo",
    "Sunderland":            "Sunderland AFC",
    "Swansea":               "Swansea City",
    "Tondela":               "CD Tondela",
    "Torino":                "Torino FC",
    "Tottenham":             "Tottenham Hotspur",
    "Toulouse":              "Toulouse FC",
    "Udinese":               "Udinese",
    "Union Berlin":          "1.FC Unión Berlín",
    "Valencia":              "Valencia CF",
    "Verona":                "Hellas Verona",
    "VfB Stuttgart":         "VfB Stuttgart",
    "Villarreal":            "Villarreal CF",
    "Vitoria de Guimaraes":  "Vitória Guimarães SC",
    "Watford":               "Watford FC",
    "Werder Bremen":         "SV Werder Bremen",
    "West Bromwich Albion":  "West Bromwich Albion",
    "West Ham":              "West Ham United",
    "Wolfsburg":             "VfL Wolfsburgo",
    "Wolves":                "Wolverhampton Wanderers",
    "Wrexham":               "Wrexham AFC",
}

In [37]:
def renombrar_equipos(df: pd.DataFrame, mapeo: dict, columna: str) -> pd.DataFrame:
    """Renombra valores en la columna indicada usando el mapeo (comparación exacta)."""
    if columna not in df.columns or not mapeo:
        return df
    antes = df[columna].copy()
    df[columna] = df[columna].replace(mapeo)
    cambios = (antes != df[columna]).sum()
    print(f"  Equipos renombrados en '{columna}': {cambios}")
    return df

Aqui construimos tablas con equipos ID, liga ID:

In [38]:
def construir_tabla_equipos(equipos_df: pd.DataFrame, mapeo: dict) -> pd.DataFrame:
    """
    Construye la tabla maestra de equipos:
      - Elimina la columna team_id antigua
      - Renombra los equipos al formato canónico (transfermarkt)
      - Crea id_equipo y id_liga
      - Deja columnas en el orden: id_equipo, id_liga, team_name, league
    """
    df = equipos_df.copy()

    # Eliminar el team_id antiguo si existe
    if "team_id" in df.columns:
        df = df.drop(columns=["team_id"])

    col_team   = next((c for c in df.columns if c.lower() in ["team", "team_name", "equipo", "name"]), None)
    col_league = next((c for c in df.columns if c.lower() in ["league", "liga"]), None)
    if col_team is None or col_league is None:
        raise ValueError(f"No se encontraron las columnas. Disponibles: {list(df.columns)}")

    # Renombrar equipos al formato canónico
    df[col_team] = df[col_team].replace(mapeo)

    # Ordenar por liga y luego por equipo para asignar IDs estables
    df = df.sort_values([col_league, col_team]).reset_index(drop=True)

    # id_liga: uno por liga única (orden alfabético)
    ligas = sorted(df[col_league].dropna().unique())
    mapa_liga = {l: i + 1 for i, l in enumerate(ligas)}
    df["id_liga"] = df[col_league].map(mapa_liga).astype("Int64")

    # id_equipo: secuencial
    df["id_equipo"] = range(1, len(df) + 1)
    df["id_equipo"] = df["id_equipo"].astype("Int64")

    # Reordenar columnas
    nuevo_orden = ["id_equipo", "id_liga", col_team, col_league]
    otras = [c for c in df.columns if c not in nuevo_orden]
    df = df[nuevo_orden + otras]

    print(f"  Tabla equipos: {len(df)} equipos, {len(ligas)} ligas")
    return df


def añadir_ids_equipo_liga(df: pd.DataFrame, tabla_equipos: pd.DataFrame,
                            col_team_origen: str,
                            col_id_equipo: str, col_id_liga: str) -> pd.DataFrame:
    """
    Añade id_equipo e id_liga a un DataFrame, ambos obtenidos por lookup del NOMBRE DEL EQUIPO
    en la tabla maestra de equipos. Las dos columnas se colocan como las primeras del DataFrame.
    """
    col_team_eq = next((c for c in tabla_equipos.columns if c.lower() in ["team", "team_name", "equipo", "name"]), None)
    if col_team_eq is None:
        raise ValueError("No se encontró columna de equipo en tabla_equipos")

    mapa_equipo  = dict(zip(tabla_equipos[col_team_eq], tabla_equipos["id_equipo"]))
    mapa_liga    = dict(zip(tabla_equipos[col_team_eq], tabla_equipos["id_liga"]))

    df[col_id_equipo] = df[col_team_origen].map(mapa_equipo).astype("Int64")
    df[col_id_liga]   = df[col_team_origen].map(mapa_liga).astype("Int64")

    sin_eq = df[col_id_equipo].isna().sum()
    sin_lg = df[col_id_liga].isna().sum()
    if sin_eq:
        print(f"  Advertencia: {sin_eq} filas sin {col_id_equipo}")
    if sin_lg:
        print(f"  Advertencia: {sin_lg} filas sin {col_id_liga}")

    # Colocar los dos IDs como primeras columnas
    cols = [c for c in df.columns if c not in (col_id_equipo, col_id_liga)]
    df = df[[col_id_equipo, col_id_liga] + cols]
    return df


def construir_tabla_liga(tabla_equipos: pd.DataFrame) -> pd.DataFrame:
    """Construye una tabla con id_liga y league a partir de la tabla maestra de equipos."""
    col_league = next((c for c in tabla_equipos.columns if c.lower() in ["league", "liga"]), None)
    if col_league is None:
        raise ValueError("No se encontró columna de liga en tabla_equipos")

    df = (
        tabla_equipos[["id_liga", col_league]]
        .drop_duplicates()
        .sort_values("id_liga")
        .reset_index(drop=True)
    )
    print(f"  Tabla liga: {len(df)} ligas")
    return df

In [43]:
def normalizar_texto_ws(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFD", x)
    x = "".join(c for c in x if unicodedata.category(c) != "Mn")
    x = " ".join(x.split())
    return x


def limpiar_numero_ws(x):
    """Convierte números tipo edad, altura, peso o minutos. Maneja separadores de miles."""
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()
    x = x.replace("cm", "").replace("kg", "").replace("%", "").strip()

    if re.match(r"^\d{1,3}([.,]\d{3})+$", x):
        x = x.replace(".", "").replace(",", "")
    else:
        x = x.replace(",", ".")

    return pd.to_numeric(x, errors="coerce")


def preparar_clave_ws_unificado(df, nombre_tabla=""):
    """Crea la clave jugador-equipo de WhoScored: nombre + equipo + edad + CM + KG."""
    df = df.copy()

    columnas_necesarias = ["nombre", "ws_team", "edad", "CM", "KG"]
    faltan = [c for c in columnas_necesarias if c not in df.columns]
    if faltan:
        raise ValueError(f"En {nombre_tabla} faltan columnas necesarias: {faltan}")

    df["_nombre_norm"] = df["nombre"].map(normalizar_texto_ws)
    df["_equipo_norm"] = df["ws_team"].map(normalizar_texto_ws)

    df["_edad_key"] = df["edad"].map(limpiar_numero_ws)
    df["_cm_key"]   = df["CM"].map(limpiar_numero_ws)
    df["_kg_key"]   = df["KG"].map(limpiar_numero_ws)

    if "ws_team_id" in df.columns:
        df["_equipo_key"] = pd.to_numeric(
            df["ws_team_id"], errors="coerce"
        ).astype("Int64").astype(str)
    else:
        df["_equipo_key"] = df["_equipo_norm"]

    df["_ws_key"] = (
        df["_nombre_norm"].astype(str) + "|" +
        df["_equipo_key"].astype(str)  + "|" +
        df["_edad_key"].astype("Int64").astype(str) + "|" +
        df["_cm_key"].astype("Int64").astype(str)   + "|" +
        df["_kg_key"].astype("Int64").astype(str)
    )

    return df

In [44]:
def construir_whoscored_unificado_limpio(whoscored, nombre_equipos, transfermarkt):
    """
    Construye una única tabla limpia de WhoScored.

    Reglas de unión de columnas:
      - Las columnas de identidad/clave ya están en la base (vienen de whoscored_jugadores).
      - Las columnas 'comunes permitidas' (ws_league, titular, suplente, Mins) se añaden
        UNA sola vez (se toman de la primera hoja donde aparezcan).
      - Las columnas que aparecen en más de una hoja con el mismo nombre pero distinto
        significado (ej: Regates) se renombran con sufijo: _defensivo, _ofensivo,
        _distribucion, _xg.
      - Las columnas únicas a una sola hoja se añaden sin sufijo.

    Fusión de transferidos:
      - Solo suma: titular, suplente, Mins.
      - El resto de datos se queda con la fila del equipo actual.
    """
    from collections import Counter

    orden_preferido = [
        "whoscore_Defensivo",
        "whoscore_Ofensivo",
        "whoscore_Distribucion",
        "whoscore_xG",
    ]

    tablas_ws = [t for t in orden_preferido if t in whoscored and t != nombre_equipos]
    tablas_ws += [t for t in whoscored.keys() if t not in tablas_ws and t != nombre_equipos]

    sufijos_por_hoja = {
        "whoscore_Defensivo":    "defensivo",
        "whoscore_Ofensivo":     "ofensivo",
        "whoscore_Distribucion": "distribucion",
        "whoscore_xG":           "xg",
    }

    columnas_identidad = [
        "_ws_key", "nombre", "ws_team", "edad", "CM", "KG",
        "posiciones", "ws_team_id", "ws_league_id",
    ]
    columnas_auxiliares = [
        "_nombre_norm", "_equipo_norm", "_edad_key",
        "_cm_key", "_kg_key", "_equipo_key",
    ]
    columnas_comunes_permitidas = {"ws_league", "titular", "suplente", "Mins","Asist","Goles","ws_tab","Rating","PClave"}

    columnas_no_estadistica = set(columnas_identidad + columnas_auxiliares) | {"_ws_key"}

    # ========================================================
    # 1. Preparar tablas WhoScored
    # ========================================================
    tablas_preparadas = {}
    for tabla in tablas_ws:
        df = whoscored[tabla].copy()
        df = preparar_clave_ws_unificado(df, nombre_tabla=tabla)
        df = df.drop_duplicates(subset=["_ws_key"], keep="first")
        tablas_preparadas[tabla] = df

    # ========================================================
    # 2. Tabla base de jugadores-equipo
    # ========================================================
    identidades = []
    for tabla, df in tablas_preparadas.items():
        cols_existentes = [c for c in columnas_identidad if c in df.columns]
        identidades.append(df[cols_existentes].copy())

    whoscored_jugadores = pd.concat(identidades, ignore_index=True)
    whoscored_jugadores = whoscored_jugadores.drop_duplicates(
        subset=["_ws_key"], keep="first"
    ).reset_index(drop=True)
    whoscored_jugadores.insert(0, "ws_player_id", range(1, len(whoscored_jugadores) + 1))

    # ========================================================
    # 3. Detectar columnas de estadística que se repiten entre hojas
    # ========================================================
    contador = Counter()
    for tabla in tablas_ws:
        for col in tablas_preparadas[tabla].columns:
            if col in columnas_no_estadistica or col in columnas_comunes_permitidas:
                continue
            contador[col] += 1
    columnas_que_se_repiten = {c for c, n in contador.items() if n > 1}

    # ========================================================
    # 4. Unir hojas: con sufijo si la columna se repite,
    #    una sola vez si es común permitida, sin sufijo si es única
    # ========================================================
    whoscored_unificado = whoscored_jugadores.copy()
    comunes_ya_anadidas = set()

    for tabla in tablas_ws:
        df = tablas_preparadas[tabla].copy()
        sufijo = sufijos_por_hoja.get(tabla, tabla.lower().replace("whoscore_", ""))

        columnas_candidatas = [
            c for c in df.columns if c not in columnas_no_estadistica
        ]
        if not columnas_candidatas:
            continue

        df_stats = df[["_ws_key"] + columnas_candidatas].copy()

        # Plan de renombrado y descarte
        rename_map = {}
        columnas_a_descartar = []

        for col in columnas_candidatas:
            if col in columnas_comunes_permitidas:
                if col in comunes_ya_anadidas:
                    columnas_a_descartar.append(col)
                else:
                    comunes_ya_anadidas.add(col)
            elif col in columnas_que_se_repiten:
                rename_map[col] = f"{col}_{sufijo}"

        if columnas_a_descartar:
            df_stats = df_stats.drop(columns=columnas_a_descartar)
        if rename_map:
            df_stats = df_stats.rename(columns=rename_map)

        whoscored_unificado = whoscored_unificado.merge(
            df_stats, on="_ws_key", how="left"
        )

    # ========================================================
    # 5. Detectar posibles transferidos
    # ========================================================
    df = whoscored_unificado.copy()
    df["_nombre_norm"]    = df["nombre"].map(normalizar_texto_ws)
    df["_edad_key"]       = df["edad"].map(limpiar_numero_ws)
    df["_cm_key"]         = df["CM"].map(limpiar_numero_ws)
    df["_kg_key"]         = df["KG"].map(limpiar_numero_ws)
    df["_ws_team_id_key"] = pd.to_numeric(df["ws_team_id"], errors="coerce").astype("Int64")
    df["_ws_team_norm"]   = df["ws_team"].map(normalizar_texto_ws)

    clave_fisica = ["_nombre_norm", "_edad_key", "_cm_key", "_kg_key"]

    ids_posibles_transferidos = set()
    for _, grupo in df.groupby(clave_fisica, dropna=False):
        if len(grupo) > 1 and grupo["_ws_team_id_key"].nunique(dropna=False) > 1:
            ids_posibles_transferidos.update(grupo["ws_player_id"].tolist())

    if len(ids_posibles_transferidos) == 0:
        whoscored_final = df.copy()
    else:
        # ====================================================
        # 6. Fusionar transferidos con ayuda de transfermarkt
        # ====================================================
        tm = transfermarkt.copy()
        columnas_tm_necesarias = ["tm_name", "tm_team", "tm_team_id"]
        faltan_tm = [c for c in columnas_tm_necesarias if c not in tm.columns]
        if faltan_tm:
            raise ValueError(f"Faltan columnas en transfermarkt: {faltan_tm}")

        tm["_nombre_norm"]    = tm["tm_name"].map(normalizar_texto_ws)
        tm["_tm_team_norm"]   = tm["tm_team"].map(normalizar_texto_ws)
        tm["_tm_team_id_key"] = pd.to_numeric(tm["tm_team_id"], errors="coerce").astype("Int64")

        filas_fusionadas = []
        ids_transferidos_fusionados = set()

        transferidos = df[df["ws_player_id"].isin(ids_posibles_transferidos)].copy()

        for _, grupo in transferidos.groupby(clave_fisica, dropna=False):
            grupo = grupo.copy()
            nombre_norm = grupo["_nombre_norm"].iloc[0]
            candidatos_tm = tm[tm["_nombre_norm"] == nombre_norm].copy()
            if len(candidatos_tm) == 0:
                continue

            equipos_ws_ids = set(grupo["_ws_team_id_key"].dropna().astype(int))
            equipos_ws_nombres = set(grupo["_ws_team_norm"].dropna())

            candidato_actual = None
            fila_actual_ws = None

            candidatos_por_id = candidatos_tm[candidatos_tm["_tm_team_id_key"].isin(equipos_ws_ids)]
            if len(candidatos_por_id) > 0:
                candidato_actual = candidatos_por_id.iloc[0]
                fila_actual_ws = grupo[
                    grupo["_ws_team_id_key"] == candidato_actual["_tm_team_id_key"]
                ].iloc[0]
            else:
                candidatos_por_nombre = candidatos_tm[
                    candidatos_tm["_tm_team_norm"].isin(equipos_ws_nombres)
                ]
                if len(candidatos_por_nombre) > 0:
                    candidato_actual = candidatos_por_nombre.iloc[0]
                    fila_actual_ws = grupo[
                        grupo["_ws_team_norm"] == candidato_actual["_tm_team_norm"]
                    ].iloc[0]

            if candidato_actual is None or fila_actual_ws is None:
                continue

            fila_base = fila_actual_ws.copy()
            for col in ["titular", "suplente", "Mins"]:
                if col in grupo.columns:
                    fila_base[col] = grupo[col].map(limpiar_numero_ws).sum(skipna=True)

            filas_fusionadas.append(fila_base)
            ids_transferidos_fusionados.update(grupo["ws_player_id"].tolist())

        if len(filas_fusionadas) > 0:
            transferidos_fusionados = pd.DataFrame(filas_fusionadas)
            base_sin_fusionados = df[~df["ws_player_id"].isin(ids_transferidos_fusionados)].copy()
            whoscored_final = pd.concat(
                [base_sin_fusionados, transferidos_fusionados],
                ignore_index=True, sort=False,
            )
        else:
            whoscored_final = df.copy()

    # ========================================================
    # 7. Limpiar columnas auxiliares
    # ========================================================
    columnas_a_eliminar = [
        "_ws_key", "_nombre_norm", "_equipo_norm",
        "_edad_key", "_cm_key", "_kg_key", "_equipo_key",
        "_ws_team_id_key", "_ws_team_norm",
    ]
    whoscored_final = whoscored_final.drop(
        columns=[c for c in columnas_a_eliminar if c in whoscored_final.columns],
        errors="ignore",
    )

    # ========================================================
    # 8. Reordenar columnas básicas al principio
    # ========================================================
    columnas_basicas = [
        "ws_player_id", "nombre", "ws_team", "edad", "CM", "KG",
        "posiciones", "ws_team_id", "ws_league_id", "ws_league",
        "titular", "suplente", "Mins",
    ]
    columnas_basicas = [c for c in columnas_basicas if c in whoscored_final.columns]
    columnas_estadisticas = [c for c in whoscored_final.columns if c not in columnas_basicas]

    whoscored_final = whoscored_final[columnas_basicas + columnas_estadisticas]
    whoscored_final = whoscored_final.reset_index(drop=True)

    return whoscored_final

In [40]:
def filtrar_transfermarkt_y_whoscored_por_nombre_equipo(
    transfermarkt,
    posiciones_valormax,
    whoscored_unificado
):
    """
    Filtra Transfermarkt, posiciones_valormax y whoscored_unificado
    para conservar únicamente jugadores que estén en Transfermarkt y
    WhoScored con el mismo nombre y equipo.

    No genera tablas auxiliares finales.
    No exporta nada.
    No añade columnas nuevas permanentes.
    """

    tm = transfermarkt.copy()
    pvm = posiciones_valormax.copy()
    ws = whoscored_unificado.copy()

    def crear_key_nombre_equipo(df, col_nombre, col_equipo_id=None, col_equipo_nombre=None):
        df = df.copy()

        df["_nombre_match"] = df[col_nombre].map(normalizar_texto_ws)

        # Preferimos usar ID de equipo si existe, porque ya viene de la tabla común de equipos
        if col_equipo_id is not None and col_equipo_id in df.columns:
            df["_equipo_match"] = pd.to_numeric(
                df[col_equipo_id],
                errors="coerce"
            ).astype("Int64").astype(str)
        else:
            df["_equipo_match"] = df[col_equipo_nombre].map(normalizar_texto_ws)

        df["_key_nombre_equipo"] = (
            df["_nombre_match"].astype(str)
            + " | "
            + df["_equipo_match"].astype(str)
        )

        return df

    # Crear claves
    tm = crear_key_nombre_equipo(
        tm,
        col_nombre="tm_name",
        col_equipo_id="tm_team_id",
        col_equipo_nombre="tm_team"
    )

    ws = crear_key_nombre_equipo(
        ws,
        col_nombre="nombre",
        col_equipo_id="ws_team_id",
        col_equipo_nombre="ws_team"
    )

    pvm = crear_key_nombre_equipo(
        pvm,
        col_nombre="tm_name",
        col_equipo_id="tm_team_id",
        col_equipo_nombre="tm_team"
    )

    # Intersección real entre Transfermarkt y WhoScored
    keys_tm = set(tm["_key_nombre_equipo"].dropna())
    keys_ws = set(ws["_key_nombre_equipo"].dropna())

    keys_comunes = keys_tm & keys_ws

    print("\nFiltro final por nombre + equipo")
    print("-" * 40)
    print(f"Transfermarkt antes:       {len(tm)}")
    print(f"WhoScored unificado antes: {len(ws)}")
    print(f"Claves comunes:            {len(keys_comunes)}")

    # Filtrar
    tm_filtrado = tm[tm["_key_nombre_equipo"].isin(keys_comunes)].copy()
    ws_filtrado = ws[ws["_key_nombre_equipo"].isin(keys_comunes)].copy()
    pvm_filtrado = pvm[pvm["_key_nombre_equipo"].isin(keys_comunes)].copy()

    print(f"Transfermarkt después:       {len(tm_filtrado)}")
    print(f"WhoScored unificado después: {len(ws_filtrado)}")
    print(f"Posiciones/valor máximo después: {len(pvm_filtrado)}")
    print(f"Eliminados de Transfermarkt:       {len(tm) - len(tm_filtrado)}")
    print(f"Eliminados de WhoScored unificado: {len(ws) - len(ws_filtrado)}")

    # Eliminar columnas auxiliares
    cols_aux = [
        "_nombre_match",
        "_equipo_match",
        "_key_nombre_equipo"
    ]

    tm_filtrado = tm_filtrado.drop(columns=[c for c in cols_aux if c in tm_filtrado.columns], errors="ignore")
    ws_filtrado = ws_filtrado.drop(columns=[c for c in cols_aux if c in ws_filtrado.columns], errors="ignore")
    pvm_filtrado = pvm_filtrado.drop(columns=[c for c in cols_aux if c in pvm_filtrado.columns], errors="ignore")

    tm_filtrado = tm_filtrado.reset_index(drop=True)
    ws_filtrado = ws_filtrado.reset_index(drop=True)
    pvm_filtrado = pvm_filtrado.reset_index(drop=True)

    return tm_filtrado, pvm_filtrado, ws_filtrado

Aqui haremos lo siguiente, le quitaremos la m (de metros) a la variable tm_height, despues las fechas de contrato la dejaremos en dd/mm/yyyy, reemplazamos - por 0 y por ultimo pasamos las variables a formato numero y rellenamos los huecos vacios por 0.


In [79]:
def limpiar_tm_height(df: pd.DataFrame, columna: str = "tm_height") -> pd.DataFrame:
    """Quita la 'm' de tm_height y deja solo el número (como float)."""
    if columna not in df.columns:
        return df
    df[columna] = (
        df[columna].astype(str)
        .str.replace("m", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df[columna] = pd.to_numeric(df[columna], errors="coerce")
    print(f"  Columna '{columna}' limpiada")
    return df


def formatear_fecha(df: pd.DataFrame, columna: str, formato_salida: str = "%d/%m/%Y") -> pd.DataFrame:
    """Convierte la columna a fecha y la formatea como dd/mm/yyyy (string)."""
    if columna not in df.columns:
        return df
    df[columna] = pd.to_datetime(df[columna], errors="coerce", dayfirst=True).dt.strftime(formato_salida)
    print(f"  Columna '{columna}' formateada a dd/mm/yyyy")
    return df


def reemplazar_guiones_por_cero(df: pd.DataFrame, columnas_excluir: list = None) -> pd.DataFrame:
    """Sustituye los '-' por 0 en todas las celdas, excepto en las columnas indicadas."""
    columnas_excluir = columnas_excluir or []
    columnas = [c for c in df.columns if c not in columnas_excluir]
    for col in columnas:
        df[col] = df[col].replace("-", 0)
    print(f"  Guiones reemplazados por 0 en {len(columnas)} columnas")
    return df


def convertir_a_numerico(df: pd.DataFrame, columnas_excluir: list = None) -> pd.DataFrame:
    """Convierte a número todas las columnas excepto las indicadas."""
    columnas_excluir = columnas_excluir or []
    for col in df.columns:
        if col in columnas_excluir:
            continue
        convertida = pd.to_numeric(
            df[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
        # Solo reemplazar si la conversión no destruye datos (al menos algún valor válido)
        if convertida.notna().any():
            df[col] = convertida
    print(f"  Columnas convertidas a numérico")
    return df

def rellenar_vacios_con_cero(df: pd.DataFrame, columnas_excluir: list = None) -> pd.DataFrame:
    """Rellena los valores vacíos (NaN) con 0 en todas las columnas excepto las indicadas."""
    columnas_excluir = columnas_excluir or []
    columnas = [c for c in df.columns if c not in columnas_excluir]
    for col in columnas:
        df[col] = df[col].fillna(0)
    print(f"  Vacíos rellenados con 0 en {len(columnas)} columnas")
    return df

Ahora vamos a unir las tablas procedentes de transfermarkt

In [88]:
def construir_datos_jugador(transfermarkt: pd.DataFrame, posiciones_valormax: pd.DataFrame) -> pd.DataFrame:
    """
    Une transfermarkt y posiciones_valormax por tm_player_id.
    No duplica columnas repetidas y excluye tm_position_squad y tm_main_position.
    Devuelve la tabla con el orden de columnas definido.
    """
    tm  = transfermarkt.copy()
    pvm = posiciones_valormax.copy()

    excluir_pvm = ["tm_position_squad", "tm_main_position"]
    cols_pvm_nuevas = ["tm_player_id"] + [
        c for c in pvm.columns if c not in tm.columns and c not in excluir_pvm
    ]
    pvm_reducida = pvm[cols_pvm_nuevas]

    datos_jugador = tm.merge(pvm_reducida, on="tm_player_id", how="left", indicator=True)

    sin_pareja = (datos_jugador["_merge"] == "left_only").sum()
    if sin_pareja:
        print(f"  Aviso: {sin_pareja} jugadores sin datos en posiciones_valormax")
    if len(datos_jugador) != len(tm):
        print(f"  Aviso: la unión generó {len(datos_jugador) - len(tm)} filas de más")

    datos_jugador = datos_jugador.drop(columns=["_merge"], errors="ignore")

    orden = [
        "tm_player_id", "tm_team_id", "tm_league_id", "tm_name", "tm_team", "tm_league",
        "tm_position",
        "pos_sec_1", "pos_sec_2", "tm_nationality", "tm_dob", "tm_age", "tm_height",
        "tm_foot", "tm_contract_until", "tm_joined", "tm_previous_club",
        "tm_market_value_M", "tm_highest_market_value_M",
    ]
    orden_existente = [c for c in orden if c in datos_jugador.columns]
    resto = [c for c in datos_jugador.columns if c not in orden_existente]
    datos_jugador = datos_jugador[orden_existente + resto]

    print(f"  Datos_Jugador: {datos_jugador.shape[0]} filas x {datos_jugador.shape[1]} columnas")
    return datos_jugador

Unimos finalmente con whoscored:

In [95]:
def construir_tabla_final(datos_jugador: pd.DataFrame, whoscored_unificado: pd.DataFrame) -> pd.DataFrame:
    """
    Une Datos_Jugador (izquierda) con whoscored_unificado (derecha)
    por nombre + id de equipo. Excluye de whoscored las columnas de identidad/equipo
    que ya vienen de Datos_Jugador.
    """
    dj = datos_jugador.copy()
    ws = whoscored_unificado.copy()

    dj["_key"] = dj["tm_name"].map(normalizar) + "|" + dj["tm_team_id"].astype("Int64").astype(str)
    ws["_key"] = ws["nombre"].map(normalizar) + "|" + ws["ws_team_id"].astype("Int64").astype(str)

    excluir_ws = [
        "ws_player_id", "nombre", "ws_team", "edad", "CM", "KG",
        "ws_team_id", "ws_league_id", "ws_league",
    ]
    cols_ws = ["_key"] + [c for c in ws.columns if c not in excluir_ws and c != "_key"]
    ws_reducida = ws[cols_ws]

    union = dj.merge(ws_reducida, on="_key", how="left", indicator=True)

    sin_pareja = (union["_merge"] == "left_only").sum()
    if sin_pareja:
        print(f"  Aviso: {sin_pareja} jugadores sin datos en whoscored")
    if len(union) != len(dj):
        print(f"  Aviso: la unión generó {len(union) - len(dj)} filas de más")

    union = union.drop(columns=["_key", "_merge"], errors="ignore")
    print(f"  tabla_final: {union.shape[0]} filas x {union.shape[1]} columnas")
    return union

Unificamos ID de transfermarlkt y de whoscored

In [106]:
def reemplazar_ws_player_id(whoscored_unificado, transfermarkt):
    """
    Sustituye los valores de ws_player_id en whoscored_unificado por el tm_player_id
    correspondiente (cruce por nombre + id de equipo).
    Elimina las filas en colisión (mismo tm_player_id para dos jugadores distintos).
    No crea columnas nuevas ni toca otras tablas.
    """
    ws = whoscored_unificado.copy()
    tm = transfermarkt.copy()

    ws["_key"] = ws["nombre"].map(normalizar) + "|" + ws["ws_team_id"].astype("Int64").astype(str)
    tm["_key"] = tm["tm_name"].map(normalizar) + "|" + tm["tm_team_id"].astype("Int64").astype(str)

    mapa_id = dict(zip(tm["_key"], tm["tm_player_id"]))
    ids = ws["_key"].map(mapa_id)

    # Detectar colisiones (mismo id asignado a más de una fila)
    colisiones = set(ids[ids.notna() & ids.duplicated(keep=False)])
    sin_match  = ids.isna().sum()

    print(f"  sin match: {sin_match}   en colisión: {len(colisiones)}")

    # Asignar y filtrar
    ws["ws_player_id"] = ids
    antes = len(ws)
    ws = ws[ws["ws_player_id"].notna() & ~ws["ws_player_id"].isin(colisiones)].copy()
    ws["ws_player_id"] = ws["ws_player_id"].astype("Int64")
    ws = ws.drop(columns=["_key"], errors="ignore")

    print(f"  filas: {antes} -> {len(ws)}")
    return ws

In [110]:
def filtrar_interseccion_por_id(transfermarkt, posiciones_valormax,
                                 datos_jugador, whoscored_unificado):
    """
    Deja en todas las tablas (que tienen tm_player_id o ws_player_id ya unificado)
    solo los jugadores presentes en TODAS ellas, usando el id de jugador.
    """
    ids_tm  = set(transfermarkt["tm_player_id"].dropna())
    ids_pvm = set(posiciones_valormax["tm_player_id"].dropna())
    ids_dj  = set(datos_jugador["tm_player_id"].dropna())
    ids_ws  = set(whoscored_unificado["ws_player_id"].dropna())  # ya contiene tm_player_id

    comunes = ids_tm & ids_pvm & ids_dj & ids_ws
    print(f"  IDs comunes a todas las tablas: {len(comunes)}")

    tm  = transfermarkt[transfermarkt["tm_player_id"].isin(comunes)].copy()
    pvm = posiciones_valormax[posiciones_valormax["tm_player_id"].isin(comunes)].copy()
    dj  = datos_jugador[datos_jugador["tm_player_id"].isin(comunes)].copy()
    ws  = whoscored_unificado[whoscored_unificado["ws_player_id"].isin(comunes)].copy()

    print(f"  Filas finales -> tm:{len(tm)}  pvm:{len(pvm)}  dj:{len(dj)}  ws:{len(ws)}")
    return tm, pvm, dj, ws

FINALMENTE RENOMBRAMOS LAS COLUMNAS EN LA TABLA FINAL PARA PODER DAR MAS CLARIDAD AL ESTUDIO:

In [117]:
def renombrar_columnas_final(df: pd.DataFrame) -> pd.DataFrame:
    """Renombra las columnas de la tabla final a nombres legibles."""
    mapeo = {
        "tm_player_id":              "PLAYER_ID",
        "tm_team_id":                "TEAM_ID",
        "tm_league_id":              "LEAGUE_ID",
        "tm_name":                   "NAME",
        "tm_team":                   "TEAM",
        "tm_league":                 "LEAGUE",
        "tm_position":               "MAIN POSITION",
        "pos_sec_1":                 "POSITION 2",
        "pos_sec_2":                 "POSITION 3",
        "tm_nationality":            "NATIONALITY",
        "tm_dob":                    "BIRTH DATE",
        "tm_age":                    "AGE",
        "tm_height":                 "HEIGHT",
        "tm_foot":                   "FOOT",
        "tm_contract_until":         "CONTRACT UNTIL",
        "tm_joined":                 "TEAM JOINED",
        "tm_previous_club":          "PREVIOUS TEAM",
        "tm_market_value_M":         "MARKET VALUE",
        "tm_highest_market_value_M": "HIGHEST MARKET VALUE",
        "titular":                   "STARTS",
        "suplente":                  "BENCH",
        "Mins":                      "MINS",
        "Entrad":                    "TACKLE",
        "Interc":                    "INTERCEPTION",
        "Falt":                      "FAULT PER MATCH",
        "FJuegoG":                   "INTENTIONAL OFFSIDES",
        "Despe":                     "CLEARANCE",
        "Rgts_defensivo":            "DRIBBLE_DEF",
        "Bloq":                      "BLOCKS",
        "Rating":                    "RATING",
        "Goles":                     "GOALS",
        "Asist":                     "ASSIST",
        "TpP":                       "SHOTS PER MATCH",
        "PClave":                    "KEY PASSES",
        "Rgts_ofensivo":             "DRIBBLE_OF",
        "FaltF":                     "TACKLE RECEIVED",
        "FJuego":                    "OFF-SIDES",
        "AP%":                       "SUCCESSFUL PASSES (%)",
        "Centr":                     "CROSS",
        "BLargos":                   "LONG PASS",
        "PHueco":                    "SPACE BALL",
        "Tiros":                     "SHOTS",
    }
    presentes = {k: v for k, v in mapeo.items() if k in df.columns}
    no_encontradas = [k for k in mapeo if k not in df.columns]

    df = df.rename(columns=presentes)
    print(f"  Columnas renombradas: {len(presentes)}")
    if no_encontradas:
        print(f"  No encontradas en la tabla (se omiten): {no_encontradas}")
    return df

# PIPELINE PRINCIPAL

In [119]:
def pipeline(config: dict) -> dict:

    # ============================================================
    # 1. CARGA DE DATOS
    # ============================================================
    print("Carga de datos")
    print("-" * 40)
    transfermarkt       = cargar_transfermarkt(config["archivo_transfermarkt"])
    posiciones_valormax = cargar_posiciones_valormax(config["archivo_posiciones_valormax"])
    whoscored           = cargar_whoscored(
        config["archivo_whoscored"],
        config["prefijo_whoscored"],
        config["nombre_hoja_final"],
    )
    nombre_equipos = config["nombre_hoja_final"]

    # ============================================================
    # 2. LIMPIEZA DE TRANSFERMARKT
    # ============================================================
    print("\nLimpieza de transfermarkt")
    print("-" * 40)
    transfermarkt = eliminar_duplicados_ignorando_col_a(transfermarkt)
    transfermarkt = eliminar_columnas(transfermarkt, ["tm_market_value_raw", "tm_dob_raw"])
    transfermarkt = limpiar_previous_club(transfermarkt)
    transfermarkt = eliminar_filas_vacias_en_columna(transfermarkt, "tm_name")
    transfermarkt = limpiar_tm_height(transfermarkt, "tm_height")
    transfermarkt = eliminar_columnas(transfermarkt, ["tm_height_cm"])
    transfermarkt = formatear_fecha(transfermarkt, "tm_dob")
    transfermarkt = formatear_fecha(transfermarkt, "tm_contract_until")
    transfermarkt = formatear_fecha(transfermarkt, "tm_joined")

    # ============================================================
    # 3. LIMPIEZA DE POSICIONES_VALORMAX
    # ============================================================
    print("\nLimpieza de posiciones_valormax")
    print("-" * 40)
    posiciones_valormax = eliminar_columnas(posiciones_valormax, [
        "tm_url",
        "tm_highest_market_value_raw",
        "tm_highest_market_value_date",
        "tm_highest_market_value_source",
        "error",
        "tm_team_id",
    ])
    posiciones_valormax = eliminar_filas_vacias_en_columna(posiciones_valormax, "tm_name")
    posiciones_valormax = separar_secondary_positions(posiciones_valormax)

    # ============================================================
    # 4. LIMPIEZA DE HOJAS WHOSCORED
    # ============================================================
    print("\nLimpieza de hojas whoscored")
    print("-" * 40)
    for nombre, df in list(whoscored.items()):
        if nombre == nombre_equipos:
            continue
        df = limpiar_hoja_whoscored(df, nombre)
        df = separar_jgdos(df)
        df = eliminar_columnas(df, ["ws_team_url", "ws_team_id"])
        df = eliminar_filas_vacias_en_columna(df, "nombre")
        df = renombrar_jugadores(df, RENOMBRES_JUGADORES)
        df = renombrar_equipos(df, RENOMBRES_EQUIPOS, "ws_team")
        if nombre == "whoscore_Defensivo":
            df = eliminar_columnas_por_prefijo(df, "Goles")
        whoscored[nombre] = df

    # ============================================================
    # 5. TABLA EQUIPOS Y TABLA LIGA
    # ============================================================
    print("\nConstrucción de la tabla equipos y tabla liga")
    print("-" * 40)
    whoscored[nombre_equipos] = eliminar_columnas(whoscored[nombre_equipos], ["team_url"])
    whoscored[nombre_equipos] = construir_tabla_equipos(whoscored[nombre_equipos], RENOMBRES_EQUIPOS)
    tabla_equipos = whoscored[nombre_equipos]
    tabla_liga    = construir_tabla_liga(tabla_equipos)

    # ============================================================
    # 6. ASIGNACIÓN DE ID DE EQUIPO Y LIGA
    # ============================================================
    print("\nAsignación de id_equipo e id_liga")
    print("-" * 40)
    transfermarkt = añadir_ids_equipo_liga(
        transfermarkt, tabla_equipos,
        col_team_origen="tm_team", col_id_equipo="tm_team_id", col_id_liga="tm_league_id",
    )
    posiciones_valormax = añadir_ids_equipo_liga(
        posiciones_valormax, tabla_equipos,
        col_team_origen="tm_team", col_id_equipo="tm_team_id", col_id_liga="tm_league_id",
    )
    for nombre, df in list(whoscored.items()):
        if nombre == nombre_equipos:
            continue
        whoscored[nombre] = añadir_ids_equipo_liga(
            df, tabla_equipos,
            col_team_origen="ws_team", col_id_equipo="ws_team_id", col_id_liga="ws_league_id",
        )

    # ============================================================
    # 7. INTERSECCIÓN DE JUGADORES POR NOMBRE
    # ============================================================
    print("\nCálculo de la intersección de jugadores")
    print("-" * 40)
    nombres_tm  = set(transfermarkt["tm_name"].map(normalizar).dropna())
    nombres_pvm = set(posiciones_valormax["tm_name"].map(normalizar).dropna())
    nombres_ws  = set()
    for nombre, df in whoscored.items():
        if nombre == nombre_equipos:
            continue
        nombres_ws.update(df["nombre"].map(normalizar).dropna())
    interseccion = nombres_tm & nombres_pvm & nombres_ws
    print(f"  transfermarkt:             {len(nombres_tm)} jugadores")
    print(f"  posiciones_valormax:       {len(nombres_pvm)} jugadores")
    print(f"  whoscored unión 4 hojas:   {len(nombres_ws)} jugadores")
    print(f"  intersección:              {len(interseccion)} jugadores")

    print("\nFiltrado por intersección")
    print("-" * 40)
    transfermarkt       = filtrar_por_interseccion(transfermarkt, interseccion, "tm_name")
    posiciones_valormax = filtrar_por_interseccion(posiciones_valormax, interseccion, "tm_name")
    for nombre, df in list(whoscored.items()):
        if nombre == nombre_equipos:
            continue
        whoscored[nombre] = filtrar_por_interseccion(df, interseccion, "nombre")

    # ============================================================
    # 8. WHOSCORED UNIFICADO
    # ============================================================
    print("\nConstrucción de whoscored_unificado")
    print("-" * 40)
    whoscored_unificado = construir_whoscored_unificado_limpio(
        whoscored=whoscored,
        nombre_equipos=nombre_equipos,
        transfermarkt=transfermarkt,
    )
    print(f"  antes del filtro final: {whoscored_unificado.shape[0]} filas x {whoscored_unificado.shape[1]} columnas")

    print("\nFiltro final: jugadores comunes Transfermarkt - WhoScored")
    print("-" * 40)
    transfermarkt, posiciones_valormax, whoscored_unificado = filtrar_transfermarkt_y_whoscored_por_nombre_equipo(
        transfermarkt=transfermarkt,
        posiciones_valormax=posiciones_valormax,
        whoscored_unificado=whoscored_unificado,
    )

    # ============================================================
    # 9. POST-PROCESAMIENTO DE WHOSCORED UNIFICADO
    # ============================================================
    print("\nPost-procesamiento de whoscored_unificado")
    print("-" * 40)
    whoscored_unificado = eliminar_columnas(
        whoscored_unificado, ["posiciones", "ws_tab", "PdasB", "Propia"]
    )
    whoscored_unificado = reemplazar_guiones_por_cero(
        whoscored_unificado, columnas_excluir=["nombre", "ws_team", "ws_league"]
    )
    whoscored_unificado = convertir_a_numerico(
        whoscored_unificado, columnas_excluir=["nombre", "ws_team", "ws_league"]
    )
    whoscored_unificado = rellenar_vacios_con_cero(
        whoscored_unificado, columnas_excluir=["nombre", "ws_team", "ws_league"]
    )

    # ============================================================
    # 10. REEMPLAZO DE ws_player_id POR tm_player_id
    # ============================================================
    print("\nReemplazo de ws_player_id por tm_player_id")
    print("-" * 40)
    whoscored_unificado = reemplazar_ws_player_id(whoscored_unificado, transfermarkt)

    # ============================================================
    # 11. DATOS_JUGADOR (transfermarkt + posiciones_valormax)
    # ============================================================
    print("\nConstrucción de Datos_Jugador")
    print("-" * 40)
    datos_jugador = construir_datos_jugador(transfermarkt, posiciones_valormax)

    # ============================================================
    # 12. FILTRADO FINAL POR ID DE JUGADOR (coherencia entre tablas)
    # ============================================================
    print("\nFiltrado final por intersección de id de jugador")
    print("-" * 40)
    transfermarkt, posiciones_valormax, datos_jugador, whoscored_unificado = filtrar_interseccion_por_id(
        transfermarkt, posiciones_valormax, datos_jugador, whoscored_unificado
    )

    # ============================================================
    # 13. TABLA FINAL (Datos_Jugador + whoscored)
    # ============================================================
    print("\nConstrucción de la tabla final")
    print("-" * 40)
    tabla_final = construir_tabla_final(datos_jugador, whoscored_unificado)
    tabla_final = renombrar_columnas_final(tabla_final)
    print("\nPipeline completado")
    return {
        "transfermarkt": transfermarkt,
        "posiciones_valormax": posiciones_valormax,
        "datos_jugador": datos_jugador,
        "whoscored_unificado": whoscored_unificado,
        "tabla_final": tabla_final,
        "equipos": tabla_equipos,
        "liga": tabla_liga,
    }

Renombramos algunos jugadores para que haga match:

# EJECUCION

In [120]:
datos = pipeline(CONFIG)

Carga de datos
----------------------------------------
transfermarkt: 6346 filas, 17 columnas
posiciones_valormax: 3766 filas, 14 columnas
whoscore_Defensivo: 4206 filas, 119 columnas
whoscore_Ofensivo: 4206 filas, 21 columnas
whoscore_Distribucion: 4206 filas, 19 columnas
whoscore_xG: 3458 filas, 18 columnas
equipos: 138 filas, 4 columnas

Limpieza de transfermarkt
----------------------------------------
Duplicados eliminados: 6346 -> 3764 (2582 filas)
Columnas eliminadas: ['tm_market_value_raw', 'tm_dob_raw']
Columna 'tm_previous_club' limpiada
  Filas eliminadas por 'tm_name' vacía: 0  (3764 -> 3764)
  Columna 'tm_height' limpiada
Columnas eliminadas: ['tm_height_cm']
  Columna 'tm_dob' formateada a dd/mm/yyyy
  Columna 'tm_contract_until' formateada a dd/mm/yyyy
  Columna 'tm_joined' formateada a dd/mm/yyyy

Limpieza de posiciones_valormax
----------------------------------------
Columnas eliminadas: ['tm_url', 'tm_highest_market_value_raw', 'tm_highest_market_value_date', 'tm_hi

# TABLAS

In [91]:
resumen = pd.DataFrame(
    [(nombre, df.shape[0], df.shape[1]) for nombre, df in datos.items()],
    columns=["Tabla", "Filas", "Columnas"]
)
display(resumen)

,Tabla,Filas,Columnas
0,transfermarkt,3407,16
1,posiciones_valormax,3407,11
2,datos_jugador,3407,19
3,whoscored_unificado,3408,38
4,equipos,138,4
5,liga,7,2


In [70]:
for nombre, df in datos.items():
    display(Markdown(f"**{nombre}**"))
    display(df.head(3))

**transfermarkt**

,tm_team_id,tm_league_id,tm_team,tm_league,tm_name,tm_player_id,tm_position,tm_dob,tm_age,tm_nationality,tm_height,tm_foot,tm_joined,tm_previous_club,tm_contract_until,tm_market_value_M
0,57,3,Real Madrid CF,La Liga,Thibaut Courtois,108390,Portero,11/05/1992,34,Bélgica,2.00,Izquierdo,09/08/2018,Chelsea FC,30/06/2027,18.0
1,57,3,Real Madrid CF,La Liga,Andriy Lunin,404839,Portero,11/02/1999,27,Ucrania,1.91,Derecho,01/07/2018,Zorya Lugansk,30/06/2030,15.0
2,57,3,Real Madrid CF,La Liga,Dean Huijsen,890290,Defensa central,14/04/2005,21,España,1.97,Ambidiestro,01/06/2025,AFC Bournemouth,30/06/2030,65.0


**posiciones_valormax**

,tm_team_id,tm_league_id,tm_league,tm_team,tm_name,tm_position_squad,tm_player_id,tm_main_position,pos_sec_1,pos_sec_2,tm_highest_market_value_M
0,57,3,La Liga,Real Madrid CF,Thibaut Courtois,Portero,108390,Portero,<NA>,<NA>,75.0
1,57,3,La Liga,Real Madrid CF,Andriy Lunin,Portero,404839,Portero,<NA>,<NA>,25.0
2,57,3,La Liga,Real Madrid CF,Dean Huijsen,Defensa central,890290,Defensa central,<NA>,<NA>,70.0


**liga**

,id_liga,league
0,1,Bundesliga
1,2,Championship
2,3,LaLiga


**equipos**

,id_equipo,id_liga,team_name,league
0,1,1,1.FC Heidenheim 1846,Bundesliga
1,2,1,1.FC Unión Berlín,Bundesliga
2,3,1,1.FSV Mainz 05,Bundesliga


**whoscored_unificado**

,ws_player_id,nombre,ws_team,edad,CM,KG,ws_team_id,ws_league_id,ws_league,titular,suplente,Mins,Entrad,Interc,Falt,FJuegoG,Despe,Rgts_defensivo,Bloq,Rating,Goles,Asist,TpP,PClave,Rgts_ofensivo,FaltF,FJuego,Despo,PromeP,AP%,Centr,BLargos,PHueco,xG,xGDif,xG/90,Tiros,xG/Tiros
0,1,Eric García,FC Barcelona,25,180,77,48,3,LaLiga,32.0,2.0,"2731,0","2,2","1,4",1,"0,6","2,3","0,7","0,2","7,02",1,2,"0,7","0,9","0,1","0,7",0,"0,3","69,1","92,2",0,"2,1","0,1","2,81","-1,81","0,09","25,0","0,11"
1,2,Frenkie de Jong,FC Barcelona,29,182,75,48,3,LaLiga,16.0,9.0,"1632,0","1,8","0,7","0,4",0,"0,6","0,4","0,2","6,86",1,5,"0,3","1,4","0,2","1,1","0,1","0,5","60,1","94,2","0,2","1,8","0,2","1,05","-0,05","0,06","7,0","0,15"
2,3,Pedri,FC Barcelona,23,174,68,48,3,LaLiga,23.0,6.0,"2110,0","1,7","0,8","0,3",0,"0,8","1,1","0,1","7,25",2,9,"0,7","2,2","1,3","1,5",0,1,"70,9","91,6","0,2","3,2","0,3","2,46","-0,46","0,1","21,0","0,12"


# exportar para revisar

In [121]:
ruta_salida = "revision_pipeline.xlsx"

with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    for nombre, df in datos.items():
        # Excel limita los nombres de hoja a 31 caracteres
        df.to_excel(writer, sheet_name=nombre[:31], index=False)

archivo_final = PROJECT_ROOT / "Estadisticas Jugadores 2025-2026.xlsx"
datos["tabla_final"].to_excel(archivo_final, index=False)

import os
print(f"Exportado en: {os.path.abspath(ruta_salida)}")
print(f"Dataset consolidado: {archivo_final.name}")
print(f"Hojas creadas: {list(datos.keys())}")

Exportado en: revision_pipeline.xlsx
Hojas creadas: ['transfermarkt', 'posiciones_valormax', 'datos_jugador', 'whoscored_unificado', 'tabla_final', 'equipos', 'liga']


# ANEXO 1. CODIGOS AUXILIARES 

# Ver cuantas coincidencias hay entre dos tablas:

In [267]:
import unicodedata

def normalizar(texto):
    """Pasa a minúsculas, quita tildes y espacios sobrantes."""
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

tm_nombres = set(datos["transfermarkt"]["tm_name"].map(normalizar).dropna())
ws_nombres = set(datos["whoscore_Defensivo"]["nombre"].map(normalizar).dropna())

matches = tm_nombres & ws_nombres
solo_tm = tm_nombres - ws_nombres
solo_ws = ws_nombres - tm_nombres

print(f"Jugadores en transfermarkt:       {len(tm_nombres)}")
print(f"Jugadores en whoscore_Defensivo:  {len(ws_nombres)}")
print(f"Coinciden exactamente:            {len(matches)}")
print(f"Solo en transfermarkt:            {len(solo_tm)}")
print(f"Solo en whoscore_Defensivo:       {len(solo_ws)}")

# Si quieres ver algunos que NO coinciden:
print("\nEjemplos solo en transfermarkt:")
for n in list(solo_tm)[:10]:
    print(f"  {n}")

print("\nEjemplos solo en whoscore_Defensivo:")
for n in list(solo_ws)[:10]:
    print(f"  {n}")

Jugadores en transfermarkt:       3402
Jugadores en whoscore_Defensivo:  3378
Coinciden exactamente:            3378
Solo en transfermarkt:            24
Solo en whoscore_Defensivo:       0

Ejemplos solo en transfermarkt:
  dan ndoye
  zach abbott
  nicolas dominguez
  callum hudson-odoi
  angus gunn
  ola aina
  jair cunha
  ibrahim sangare
  stefan ortega
  omari hutchinson

Ejemplos solo en whoscore_Defensivo:


In [268]:
import unicodedata
from itertools import combinations

def normalizar(texto):
    if pd.isna(texto):
        return None
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

# Cada tabla con la columna que contiene el nombre del jugador
mapping = {
    "transfermarkt":         "tm_name",
    "posiciones_valormax":   "tm_name",
    "whoscore_Defensivo":    "nombre",
    "whoscore_Ofensivo":     "nombre",
    "whoscore_Distribucion": "nombre",
    "whoscore_xG":           "nombre",
}

# Construir conjuntos normalizados
conjuntos = {}
for tabla, columna in mapping.items():
    if tabla in datos and columna in datos[tabla].columns:
        conjuntos[tabla] = set(datos[tabla][columna].map(normalizar).dropna())

# Jugadores únicos por tabla
print("Jugadores únicos por tabla")
print("-" * 40)
for tabla, conjunto in conjuntos.items():
    print(f"  {tabla:<25} {len(conjunto)}")

# Matriz de intersecciones entre tablas
nombres = list(conjuntos.keys())
matriz = pd.DataFrame(index=nombres, columns=nombres, dtype=int)
for a in nombres:
    for b in nombres:
        matriz.loc[a, b] = len(conjuntos[a] & conjuntos[b])

print("\nMatriz de coincidencias (intersección de cada par de tablas)")
display(matriz)

# Intersección común a todas las tablas
comun_a_todas = set.intersection(*conjuntos.values())
print(f"\nJugadores presentes en TODAS las tablas: {len(comun_a_todas)}")

# Detalle de pares
print("\nDetalle por pares")
print("-" * 40)
for a, b in combinations(nombres, 2):
    inter = conjuntos[a] & conjuntos[b]
    print(f"  {a:<25} ∩ {b:<25} {len(inter)}")

Jugadores únicos por tabla
----------------------------------------
  transfermarkt             3402
  posiciones_valormax       3402
  whoscore_Defensivo        3378
  whoscore_Ofensivo         3402
  whoscore_Distribucion     3402
  whoscore_xG               2965

Matriz de coincidencias (intersección de cada par de tablas)


,transfermarkt,posiciones_valormax,whoscore_Defensivo,whoscore_Ofensivo,whoscore_Distribucion,whoscore_xG
transfermarkt,3402.0,3402.0,3378.0,3402.0,3402.0,2965.0
posiciones_valormax,3402.0,3402.0,3378.0,3402.0,3402.0,2965.0
whoscore_Defensivo,3378.0,3378.0,3378.0,3378.0,3378.0,2946.0
whoscore_Ofensivo,3402.0,3402.0,3378.0,3402.0,3402.0,2965.0
whoscore_Distribucion,3402.0,3402.0,3378.0,3402.0,3402.0,2965.0
whoscore_xG,2965.0,2965.0,2946.0,2965.0,2965.0,2965.0



Jugadores presentes en TODAS las tablas: 2946

Detalle por pares
----------------------------------------
  transfermarkt             ∩ posiciones_valormax       3402
  transfermarkt             ∩ whoscore_Defensivo        3378
  transfermarkt             ∩ whoscore_Ofensivo         3402
  transfermarkt             ∩ whoscore_Distribucion     3402
  transfermarkt             ∩ whoscore_xG               2965
  posiciones_valormax       ∩ whoscore_Defensivo        3378
  posiciones_valormax       ∩ whoscore_Ofensivo         3402
  posiciones_valormax       ∩ whoscore_Distribucion     3402
  posiciones_valormax       ∩ whoscore_xG               2965
  whoscore_Defensivo        ∩ whoscore_Ofensivo         3378
  whoscore_Defensivo        ∩ whoscore_Distribucion     3378
  whoscore_Defensivo        ∩ whoscore_xG               2946
  whoscore_Ofensivo         ∩ whoscore_Distribucion     3402
  whoscore_Ofensivo         ∩ whoscore_xG               2965
  whoscore_Distribucion     ∩ whoscore_

NO MATCHES Y POSIBLES MATCHES

In [275]:
tablas_ws = ["whoscore_Defensivo", "whoscore_Ofensivo", "whoscore_Distribucion", "whoscore_xG"]

problematicos_por_tabla = {}

print("Jugadores con mismo nombre pero CM o KG distintos")
print("-" * 60)

for tabla in tablas_ws:
    df = datos[tabla].copy()
    df["_nombre_norm"] = df["nombre"].map(normalizar)

    # Quedarnos con los nombres que aparecen más de una vez
    mascara_dup = df["_nombre_norm"].duplicated(keep=False)
    duplicados = df[mascara_dup]

    # Por cada grupo de mismo nombre, comprobar si CM o KG difieren
    grupos_problematicos = []
    for nombre_norm, grupo in duplicados.groupby("_nombre_norm"):
        cm_distintos = grupo["CM"].nunique(dropna=False) > 1
        kg_distintos = grupo["KG"].nunique(dropna=False) > 1
        if cm_distintos or kg_distintos:
            grupos_problematicos.append(grupo)

    if grupos_problematicos:
        df_problem = pd.concat(grupos_problematicos).sort_values("_nombre_norm")
        problematicos_por_tabla[tabla] = df_problem.drop(columns=["_nombre_norm"])
        n_jugadores = df_problem["_nombre_norm"].nunique()
        print(f"  {tabla:<25} {n_jugadores} jugadores problemáticos ({len(df_problem)} filas)")
    else:
        problematicos_por_tabla[tabla] = pd.DataFrame()
        print(f"  {tabla:<25} 0 jugadores problemáticos")

# Exportar
ruta_salida = "mismo_nombre_distinto_fisico.xlsx"
with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    for tabla, df_problem in problematicos_por_tabla.items():
        if len(df_problem) > 0:
            df_problem.to_excel(writer, sheet_name=tabla[:31], index=False)
        else:
            pd.DataFrame({"info": ["sin jugadores problemáticos"]}).to_excel(
                writer, sheet_name=tabla[:31], index=False
            )

import os
print(f"\nExportado en: {os.path.abspath(ruta_salida)}")

Jugadores con mismo nombre pero CM o KG distintos
------------------------------------------------------------
  whoscore_Defensivo        19 jugadores problemáticos (39 filas)
  whoscore_Ofensivo         19 jugadores problemáticos (39 filas)
  whoscore_Distribucion     19 jugadores problemáticos (39 filas)
  whoscore_xG               10 jugadores problemáticos (21 filas)

Exportado en: mismo_nombre_distinto_fisico.xlsx


In [219]:
equipos_tm = (
    datos["transfermarkt"]["tm_team"]
    .value_counts(dropna=False)
    .reset_index()
)
equipos_tm.columns = ["equipo", "n_jugadores"]

print(f"Equipos distintos en transfermarkt: {len(equipos_tm)}")
display(equipos_tm)
equipos_tm.to_excel("equipos_transfermarkt.xlsx", index=False)
import os
print(f"Exportado en: {os.path.abspath('equipos_transfermarkt.xlsx')}")

Equipos distintos en transfermarkt: 138


,equipo,n_jugadores
0,Portsmouth FC,33
1,Norwich City,32
2,Blackburn Rovers,30
3,Pisa Sporting Club,29
4,Sheffield United,29
...,...,...
133,Toulouse FC,21
134,CD Nacional,21
135,FC Famalicão,21
136,CD Tondela,21


Exportado en: equipos_transfermarkt.xlsx


TRATAMOS A FONDO LOS DUPLICADOS

En transfermarkt no tenemos problemas, todos tienen un ID diferente.

In [266]:
tm  = datos["transfermarkt"]
pvm = datos["posiciones_valormax"]

# 1. Unicidad dentro de cada tabla
print("Unicidad de tm_player_id")
print("-" * 40)
print(f"transfermarkt:        {len(tm)} filas, {tm['tm_player_id'].nunique()} IDs únicos, "
      f"{tm['tm_player_id'].duplicated().sum()} duplicados, {tm['tm_player_id'].isna().sum()} vacíos")
print(f"posiciones_valormax:  {len(pvm)} filas, {pvm['tm_player_id'].nunique()} IDs únicos, "
      f"{pvm['tm_player_id'].duplicated().sum()} duplicados, {pvm['tm_player_id'].isna().sum()} vacíos")

# 2. Comparación entre tablas
ids_tm  = set(tm['tm_player_id'].dropna())
ids_pvm = set(pvm['tm_player_id'].dropna())

solo_tm  = ids_tm - ids_pvm
solo_pvm = ids_pvm - ids_tm
comunes  = ids_tm & ids_pvm

print("\nCoincidencia entre tablas")
print("-" * 40)
print(f"IDs comunes:                 {len(comunes)}")
print(f"Solo en transfermarkt:       {len(solo_tm)}")
print(f"Solo en posiciones_valormax: {len(solo_pvm)}")

if solo_tm:
    print(f"\nEjemplos solo en transfermarkt: {sorted(solo_tm)[:10]}")
if solo_pvm:
    print(f"\nEjemplos solo en posiciones_valormax: {sorted(solo_pvm)[:10]}")

Unicidad de tm_player_id
----------------------------------------
transfermarkt:        3416 filas, 3416 IDs únicos, 0 duplicados, 0 vacíos
posiciones_valormax:  3416 filas, 3416 IDs únicos, 0 duplicados, 0 vacíos

Coincidencia entre tablas
----------------------------------------
IDs comunes:                 3416
Solo en transfermarkt:       0
Solo en posiciones_valormax: 0


Veamos como proceder en whoscore, primero vamos a obtener un excel con 8 hojas, dos por cada hoja de whoscore la primera un excel con los jugadores que se repite tanto el nombre como la altura, de esta manera identificamos los jugadores que han estado en dos equipos esta temporada (a menos de que haya un jugador con mismo nombre y altura que otro), por si acaso cuando me hagas esta tabla me vas a poner cuantos registros hay de cada jugador asi si veo que de uno hay 3 o 4 me oleria mal. En la siguiente hoja, me vas a poner una tabla con todos los jugadores que se llaman igual MENOS los que estaban en la otra tabla, de esta manera me quedaran los que tienen mismo nombre que otro jugador pero no esta en la tabla de jugadores con dos equipos por temporada, todo esto sin pipeline

In [279]:
tablas_ws = ["whoscore_Defensivo", "whoscore_Ofensivo", "whoscore_Distribucion", "whoscore_xG"]

resultado = {}
claves = ["_norm", "CM", "KG", "edad"]

for tabla in tablas_ws:
    df = datos[tabla].copy()
    df["_norm"] = df["nombre"].map(normalizar)

    # Tabla A: mismo nombre, altura, peso y edad (mismo jugador en dos equipos)
    mask_a = df.duplicated(subset=claves, keep=False)
    tabla_a = df[mask_a].copy()

    if len(tabla_a) > 0:
        conteo = tabla_a.groupby(claves).size().rename("n_filas").reset_index()
        tabla_a = tabla_a.merge(conteo, on=claves, how="left")
        tabla_a = tabla_a.sort_values(claves)

    # Tabla B: mismo nombre pero alguno de los otros campos distinto (homónimos)
    mask_nombre_repetido = df.duplicated(subset=["_norm"], keep=False)
    mask_b = mask_nombre_repetido & ~mask_a
    tabla_b = df[mask_b].copy()
    if len(tabla_b) > 0:
        tabla_b = tabla_b.sort_values(claves)

    n_jug_a = tabla_a["_norm"].nunique() if len(tabla_a) > 0 else 0
    n_jug_b = tabla_b["_norm"].nunique() if len(tabla_b) > 0 else 0
    print(f"{tabla}")
    print(f"  cambio de equipo:  {n_jug_a} jugadores ({len(tabla_a)} filas)")
    print(f"  homónimos:         {n_jug_b} jugadores ({len(tabla_b)} filas)")

    if "_norm" in tabla_a.columns:
        tabla_a = tabla_a.drop(columns=["_norm"])
    if "_norm" in tabla_b.columns:
        tabla_b = tabla_b.drop(columns=["_norm"])
    if "n_filas" in tabla_a.columns:
        cols = ["n_filas"] + [c for c in tabla_a.columns if c != "n_filas"]
        tabla_a = tabla_a[cols]

    resultado[tabla] = {"cambio_equipo": tabla_a, "homonimos": tabla_b}

# Exportar
ruta_salida = "whoscored_duplicados_analisis.xlsx"
with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    for tabla, dfs in resultado.items():
        nombre_corto = tabla.replace("whoscore_", "")
        dfs["cambio_equipo"].to_excel(writer, sheet_name=f"{nombre_corto}_cambio"[:31], index=False)
        dfs["homonimos"].to_excel(writer, sheet_name=f"{nombre_corto}_homonimos"[:31], index=False)

import os
print(f"\nExportado en: {os.path.abspath(ruta_salida)}")

whoscore_Defensivo
  cambio de equipo:  239 jugadores (478 filas)
  homónimos:         21 jugadores (40 filas)
whoscore_Ofensivo
  cambio de equipo:  244 jugadores (488 filas)
  homónimos:         21 jugadores (40 filas)
whoscore_Distribucion
  cambio de equipo:  244 jugadores (488 filas)
  homónimos:         21 jugadores (40 filas)
whoscore_xG
  cambio de equipo:  185 jugadores (370 filas)
  homónimos:         11 jugadores (20 filas)

Exportado en: whoscored_duplicados_analisis.xlsx


# COMENZAMOS A UNIR TABLAS

In [87]:
tm  = datos["transfermarkt"].copy()
pvm = datos["posiciones_valormax"].copy()

# 1. Unicidad del ID
print("Unicidad de tm_player_id")
print("-" * 50)
print(f"transfermarkt:        {len(tm)} filas, {tm['tm_player_id'].duplicated().sum()} duplicados")
print(f"posiciones_valormax:  {len(pvm)} filas, {pvm['tm_player_id'].duplicated().sum()} duplicados")

# 2. Columnas a excluir de posiciones_valormax
excluir_pvm = ["tm_position_squad", "tm_main_position"]

# De pvm: solo el ID + columnas nuevas (no repetidas en tm) y no excluidas
cols_pvm_nuevas = ["tm_player_id"] + [
    c for c in pvm.columns
    if c not in tm.columns and c not in excluir_pvm
]
pvm_reducida = pvm[cols_pvm_nuevas]

cols_repetidas = [c for c in pvm.columns if c in tm.columns and c != "tm_player_id"]
print(f"\nColumnas repetidas no duplicadas: {cols_repetidas}")
print(f"Columnas excluidas de pvm:        {excluir_pvm}")

# 3. Merge por ID
tm_pvm = tm.merge(pvm_reducida, on="tm_player_id", how="left", indicator=True)

# 4. Verificar filas
print("\nResultado de la unión")
print("-" * 50)
print(f"Filas en transfermarkt: {len(tm)}")
print(f"Filas tras la unión:    {len(tm_pvm)}")
if len(tm_pvm) == len(tm):
    print("OK: una única unión por jugador, sin duplicados.")
else:
    print(f"AVISO: {len(tm_pvm) - len(tm)} filas de más.")

# 5. Sin pareja
no_unidos = tm_pvm[tm_pvm["_merge"] == "left_only"]
print(f"\nSin datos en posiciones_valormax: {len(no_unidos)}")
if len(no_unidos) > 0:
    display(no_unidos[["tm_player_id", "tm_name", "tm_team"]].reset_index(drop=True))

tm_pvm = tm_pvm.drop(columns=["_merge"], errors="ignore")

# 6. Reordenar columnas
orden = [
    "tm_player_id", "tm_team_id", "tm_league_id", "tm_name", "tm_team", "tm_league",
    "tm_position",
    "pos_sec_1", "pos_sec_2", "tm_nationality", "tm_dob", "tm_age", "tm_height",
    "tm_foot", "tm_contract_until", "tm_joined", "tm_previous_club",
    "tm_market_value_M", "tm_highest_market_value_M",
]
orden_existente = [c for c in orden if c in tm_pvm.columns]
resto = [c for c in tm_pvm.columns if c not in orden_existente]
tm_pvm = tm_pvm[orden_existente + resto]

faltan = [c for c in orden if c not in tm_pvm.columns]
if faltan:
    print(f"\nAviso: columnas del orden que no existen en la tabla: {faltan}")
if resto:
    print(f"Columnas no incluidas en el orden (van al final): {resto}")

# 7. Exportar
ruta_salida = "union_transfermarkt_posiciones_ordenado.xlsx"
tm_pvm.to_excel(ruta_salida, index=False)
import os
print(f"\nExportado en: {os.path.abspath(ruta_salida)}")
print(f"Columnas finales ({len(tm_pvm.columns)}): {list(tm_pvm.columns)}")

Unicidad de tm_player_id
--------------------------------------------------
transfermarkt:        3407 filas, 0 duplicados
posiciones_valormax:  3407 filas, 0 duplicados

Columnas repetidas no duplicadas: ['tm_team_id', 'tm_league_id', 'tm_league', 'tm_team', 'tm_name']
Columnas excluidas de pvm:        ['tm_position_squad', 'tm_main_position']

Resultado de la unión
--------------------------------------------------
Filas en transfermarkt: 3407
Filas tras la unión:    3407
OK: una única unión por jugador, sin duplicados.

Sin datos en posiciones_valormax: 0

Exportado en: union_transfermarkt_posiciones_ordenado.xlsx
Columnas finales (19): ['tm_player_id', 'tm_team_id', 'tm_league_id', 'tm_name', 'tm_team', 'tm_league', 'tm_position', 'pos_sec_1', 'pos_sec_2', 'tm_nationality', 'tm_dob', 'tm_age', 'tm_height', 'tm_foot', 'tm_contract_until', 'tm_joined', 'tm_previous_club', 'tm_market_value_M', 'tm_highest_market_value_M']


In [94]:
dj = datos["datos_jugador"].copy()
ws = datos["whoscored_unificado"].copy()

# Clave de cruce: nombre normalizado + id de equipo
dj["_key"] = dj["tm_name"].map(normalizar) + "|" + dj["tm_team_id"].astype("Int64").astype(str)
ws["_key"] = ws["nombre"].map(normalizar) + "|" + ws["ws_team_id"].astype("Int64").astype(str)

# 1. Unicidad de la clave
print("Unicidad de la clave (nombre + id_equipo)")
print("-" * 50)
print(f"Datos_Jugador:        {len(dj)} filas, {dj['_key'].duplicated().sum()} claves duplicadas")
print(f"whoscored_unificado:  {len(ws)} filas, {ws['_key'].duplicated().sum()} claves duplicadas")

# 2. Columnas de whoscored a excluir
excluir_ws = [
    "ws_player_id", "nombre", "ws_team", "edad", "CM", "KG",
    "ws_team_id", "ws_league_id", "ws_league",
]
cols_ws = ["_key"] + [c for c in ws.columns if c not in excluir_ws and c != "_key"]
ws_reducida = ws[cols_ws]

# 3. Merge a la izquierda (todos los Datos_Jugador)
union = dj.merge(ws_reducida, on="_key", how="left", indicator=True)

# 4. Verificar filas
print("\nResultado de la unión")
print("-" * 50)
print(f"Filas en Datos_Jugador: {len(dj)}")
print(f"Filas tras la unión:    {len(union)}")
if len(union) == len(dj):
    print("OK: una única unión por jugador, sin duplicados.")
else:
    print(f"AVISO: {len(union) - len(dj)} filas de más; algún jugador casó con varias filas de whoscored.")

# 5. Jugadores sin datos de whoscored
no_unidos = union[union["_merge"] == "left_only"]
print(f"\nJugadores de Datos_Jugador sin datos en whoscored: {len(no_unidos)}")
if len(no_unidos) > 0:
    display(no_unidos[["tm_name", "tm_team", "tm_team_id"]].reset_index(drop=True))

# 6. Limpiar y exportar
union = union.drop(columns=["_key", "_merge"], errors="ignore")

ruta_salida = "union_final.xlsx"
union.to_excel(ruta_salida, index=False)
import os
print(f"\nExportado en: {os.path.abspath(ruta_salida)}")
print(f"Columnas finales: {len(union.columns)}")

Unicidad de la clave (nombre + id_equipo)
--------------------------------------------------
Datos_Jugador:        3407 filas, 0 claves duplicadas
whoscored_unificado:  3408 filas, 1 claves duplicadas

Resultado de la unión
--------------------------------------------------
Filas en Datos_Jugador: 3407
Filas tras la unión:    3408
AVISO: 1 filas de más; algún jugador casó con varias filas de whoscored.

Jugadores de Datos_Jugador sin datos en whoscored: 0

Exportado en: union_final.xlsx
Columnas finales: 48


In [101]:
ws = datos["whoscored_unificado"].copy()
tm = datos["transfermarkt"].copy()

# Reconstruir las claves
ws["_key"] = ws["nombre"].map(normalizar) + "|" + ws["ws_team_id"].astype("Int64").astype(str)
tm["_key"] = tm["tm_name"].map(normalizar) + "|" + tm["tm_team_id"].astype("Int64").astype(str)
mapa_id = dict(zip(tm["_key"], tm["tm_player_id"]))
ws["tm_player_id"] = ws["_key"].map(mapa_id)

# === IDENTIFICAR PROBLEMÁTICOS ===
# a) jugadores de whoscored sin match -> los identificamos por su _key de whoscored
keys_sin_match = set(ws.loc[ws["tm_player_id"].isna(), "_key"])

# b) tm_player_id asignados a más de un jugador de whoscored (colisión de identidad)
asignados = ws[ws["tm_player_id"].notna()]
ids_repetidos = set(
    asignados.loc[asignados["tm_player_id"].duplicated(keep=False), "tm_player_id"]
)

print("Problemáticos detectados")
print("-" * 50)
print(f"whoscored sin match (por _key):       {len(keys_sin_match)}")
print(f"tm_player_id en colisión:             {len(ids_repetidos)}")

# === ELIMINAR DE TODAS LAS TABLAS ===
# 1) De whoscored_unificado: quitar filas sin match y filas con id en colisión
ws_limpio = ws[
    (ws["tm_player_id"].notna()) &
    (~ws["tm_player_id"].isin(ids_repetidos))
].copy()
filas_ws_elim = len(ws) - len(ws_limpio)

# IDs problemáticos resueltos a tm_player_id (los de colisión sí tienen id)
ids_a_eliminar = set(ids_repetidos)

# 2) De transfermarkt, posiciones_valormax y Datos_Jugador: quitar por tm_player_id
def quitar_por_id(df, ids):
    antes = len(df)
    df2 = df[~df["tm_player_id"].isin(ids)].copy()
    return df2, antes - len(df2)

tm_limpio,  e_tm  = quitar_por_id(datos["transfermarkt"], ids_a_eliminar)
pvm_limpio, e_pvm = quitar_por_id(datos["posiciones_valormax"], ids_a_eliminar)
dj_limpio,  e_dj  = quitar_por_id(datos["datos_jugador"], ids_a_eliminar)

# Limpiar auxiliares de ws
ws_limpio = ws_limpio.drop(columns=["_key"], errors="ignore")

print("\nFilas eliminadas por tabla")
print("-" * 50)
print(f"whoscored_unificado: {filas_ws_elim}")
print(f"transfermarkt:       {e_tm}")
print(f"posiciones_valormax: {e_pvm}")
print(f"Datos_Jugador:       {e_dj}")

# Mostrar quiénes eran los de colisión
if ids_a_eliminar:
    print("\nJugadores eliminados por colisión de id:")
    display(datos["transfermarkt"][datos["transfermarkt"]["tm_player_id"].isin(ids_a_eliminar)]
            [["tm_player_id", "tm_name", "tm_team"]].reset_index(drop=True))

Problemáticos detectados
--------------------------------------------------
whoscored sin match (por _key):       0
tm_player_id en colisión:             1

Filas eliminadas por tabla
--------------------------------------------------
whoscored_unificado: 2
transfermarkt:       1
posiciones_valormax: 1
Datos_Jugador:       1

Jugadores eliminados por colisión de id:


,tm_player_id,tm_name,tm_team
0,365108,Dan Ndoye,Nottingham Forest
